# Qwen Base Model: MLP Tensor Disentanglement and MMLU

Loads `Qwen/Qwen3.5-0.8B-Base`, prints compact model structure summaries, runs a baseline MMLU evaluation, applies tensor disentanglement to selected MLP linear layers, then reruns MMLU and compares the scores.

In [1]:
import contextlib
import gc
import io
import logging
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"

# Override this if needed: "cuda", "mps", or "cpu".
REQUESTED_DEVICE = None

if REQUESTED_DEVICE is not None:
    DEVICE = REQUESTED_DEVICE
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    TORCH_DTYPE = torch.bfloat16
elif DEVICE == "mps":
    # fp32 + lm-eval auto batching can exceed unified memory on Apple Silicon.
    TORCH_DTYPE = torch.float16
else:
    TORCH_DTYPE = torch.float32

print(f"Loading {MODEL_NAME} on {DEVICE} with dtype={TORCH_DTYPE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs = {
    "dtype": TORCH_DTYPE,
    "trust_remote_code": True,
}
if DEVICE == "cuda":
    load_kwargs["device_map"] = "cuda"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
if DEVICE != "cuda":
    model = model.to(DEVICE)

model.eval()
model.config.use_cache = False
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

print(f"Loaded model class: {model.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading Qwen/Qwen3.5-0.8B-Base on mps with dtype=torch.float16


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded model class: Qwen3_5ForCausalLM
Parameters: 752,393,024


In [2]:
from collections import Counter

import torch.nn as nn
from IPython.display import Markdown, display


def count_params(module, *, recurse=True):
    return sum(param.numel() for param in module.parameters(recurse=recurse))


def fmt_int(value):
    return f"{value:,}" if isinstance(value, int) else value


def md_table(headers, rows):
    rows = [[str(cell) for cell in row] for row in rows]
    header = "| " + " | ".join(headers) + " |"
    sep = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, sep, *body])


layers = list(model.model.layers)
layer_types = getattr(model.config, "layer_types", [getattr(layer, "layer_type", "unknown") for layer in layers])
layer_type_counts = Counter(layer_types)

summary_rows = [
    ["model", MODEL_NAME],
    ["class", model.__class__.__name__],
    ["device", DEVICE],
    ["dtype", str(TORCH_DTYPE).replace("torch.", "")],
    ["parameters", fmt_int(count_params(model))],
    ["hidden size", getattr(model.config, "hidden_size", None)],
    ["intermediate size", getattr(model.config, "intermediate_size", None)],
    ["decoder layers", len(layers)],
    ["attention heads", getattr(model.config, "num_attention_heads", None)],
    ["vocab size", fmt_int(getattr(model.config, "vocab_size", None))],
    ["layer types", ", ".join(f"{name}: {count}" for name, count in layer_type_counts.items())],
]
display(Markdown("## Model summary\n" + md_table(["field", "value"], summary_rows)))

top_modules = [
    ["model.embed_tokens", model.model.embed_tokens.__class__.__name__, fmt_int(count_params(model.model.embed_tokens))],
    ["model.layers", model.model.layers.__class__.__name__, fmt_int(count_params(model.model.layers))],
    ["model.norm", model.model.norm.__class__.__name__, fmt_int(count_params(model.model.norm))],
    ["lm_head", model.lm_head.__class__.__name__, fmt_int(count_params(model.lm_head))],
]
display(Markdown("## Top-level modules\n" + md_table(["module", "type", "params"], top_modules)))

layer_rows = []
for idx, layer in enumerate(layers):
    if hasattr(layer, "linear_attn"):
        mixer_name = "linear_attn"
        mixer = layer.linear_attn
    elif hasattr(layer, "self_attn"):
        mixer_name = "self_attn"
        mixer = layer.self_attn
    else:
        mixer_name = "unknown"
        mixer = layer
    layer_rows.append([
        idx,
        layer_types[idx] if idx < len(layer_types) else getattr(layer, "layer_type", "unknown"),
        mixer_name,
        mixer.__class__.__name__,
        fmt_int(count_params(mixer)),
        fmt_int(count_params(layer.mlp)),
    ])
display(Markdown("## Decoder layers\n" + md_table(["idx", "layer type", "mixer", "mixer class", "mixer params", "mlp params"], layer_rows)))

mlp_shape_counts = Counter()
for layer in layers:
    for name, submodule in layer.mlp.named_modules():
        if isinstance(submodule, nn.Linear):
            mlp_shape_counts[(name, tuple(submodule.weight.shape), submodule.bias is not None)] += 1

mlp_rows = [
    [name, f"{shape[0]} x {shape[1]}", has_bias, count]
    for (name, shape, has_bias), count in sorted(mlp_shape_counts.items())
]
display(Markdown("## MLP linear shapes\n" + md_table(["MLP submodule", "weight shape", "bias", "count"], mlp_rows)))

## Model summary
| field | value |
| --- | --- |
| model | Qwen/Qwen3.5-0.8B-Base |
| class | Qwen3_5ForCausalLM |
| device | mps |
| dtype | float16 |
| parameters | 752,393,024 |
| hidden size | 1024 |
| intermediate size | 3584 |
| decoder layers | 24 |
| attention heads | 8 |
| vocab size | 248,320 |
| layer types | linear_attention: 18, full_attention: 6 |

## Top-level modules
| module | type | params |
| --- | --- | --- |
| model.embed_tokens | Embedding | 254,279,680 |
| model.layers | ModuleList | 498,112,320 |
| model.norm | Qwen3_5RMSNorm | 1,024 |
| lm_head | Linear | 254,279,680 |

## Decoder layers
| idx | layer type | mixer | mixer class | mixer params | mlp params |
| --- | --- | --- | --- | --- | --- |
| 0 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 1 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 2 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 3 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |
| 4 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 5 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 6 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 7 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |
| 8 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 9 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 10 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 11 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |
| 12 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 13 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 14 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 15 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |
| 16 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 17 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 18 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 19 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |
| 20 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 21 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 22 | linear_attention | linear_attn | Qwen3_5GatedDeltaNet | 10,543,264 | 11,010,048 |
| 23 | full_attention | self_attn | Qwen3_5Attention | 7,340,544 | 11,010,048 |

## MLP linear shapes
| MLP submodule | weight shape | bias | count |
| --- | --- | --- | --- |
| down_proj | 1024 x 3584 | False | 24 |
| gate_proj | 3584 x 1024 | False | 24 |
| up_proj | 3584 x 1024 | False | 24 |

In [3]:
import gc

from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

NUM_FEWSHOT = 5
EVAL_LIMIT = 5  # Use None for the full MMLU run once the smoke test works.
TASKS = ["mmlu"]
EVAL_BATCH_SIZE = 1
MAX_LENGTH = 2048

EVAL_RUNS = {}


def clear_device_cache():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def first_metric(metrics, names):
    for name in names:
        if name in metrics and isinstance(metrics[name], (int, float)):
            return float(metrics[name])
    return None


def extract_mmlu_scores(results):
    result_dict = results["results"]
    if "mmlu" in result_dict:
        metrics = result_dict["mmlu"]
        return {
            "acc": first_metric(metrics, ["acc,none", "acc"]),
            "acc_stderr": first_metric(metrics, ["acc_stderr,none", "acc_stderr"]),
            "acc_norm": first_metric(metrics, ["acc_norm,none", "acc_norm"]),
            "acc_norm_stderr": first_metric(metrics, ["acc_norm_stderr,none", "acc_norm_stderr"]),
            "num_tasks": 1,
        }

    acc_values = []
    acc_norm_values = []
    for metrics in result_dict.values():
        if not isinstance(metrics, dict):
            continue
        acc = first_metric(metrics, ["acc,none", "acc"])
        acc_norm = first_metric(metrics, ["acc_norm,none", "acc_norm"])
        if acc is not None:
            acc_values.append(acc)
        if acc_norm is not None:
            acc_norm_values.append(acc_norm)
    return {
        "acc": sum(acc_values) / len(acc_values) if acc_values else None,
        "acc_stderr": None,
        "acc_norm": sum(acc_norm_values) / len(acc_norm_values) if acc_norm_values else None,
        "acc_norm_stderr": None,
        "num_tasks": len(result_dict),
    }


def fmt_float(value):
    return "n/a" if value is None else f"{value:.4f}"


def display_eval_scores(label, scores):
    rows = [
        ["acc", fmt_float(scores["acc"])],
        ["acc stderr", fmt_float(scores["acc_stderr"])],
        ["acc norm", fmt_float(scores["acc_norm"])],
        ["acc norm stderr", fmt_float(scores["acc_norm_stderr"])],
        ["reported tasks", scores["num_tasks"]],
    ]
    display(Markdown(f"## {label} MMLU summary\n" + md_table(["metric", "value"], rows)))


def run_mmlu_eval(label):
    clear_device_cache()
    lm = HFLM(
        pretrained=model,
        tokenizer=tokenizer,
        batch_size=EVAL_BATCH_SIZE,
        max_batch_size=EVAL_BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
        dtype=TORCH_DTYPE,
        trust_remote_code=True,
    )
    print(f"Running {label} MMLU: tasks={TASKS}, fewshot={NUM_FEWSHOT}, limit={EVAL_LIMIT}, batch={EVAL_BATCH_SIZE}")
    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()
    previous_logging_disable_level = logging.root.manager.disable
    try:
        logging.disable(logging.WARNING)
        with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
            results = evaluator.simple_evaluate(
                model=lm,
                tasks=TASKS,
                num_fewshot=NUM_FEWSHOT,
                limit=EVAL_LIMIT,
                log_samples=False,
                bootstrap_iters=1000,
                verbosity="ERROR",
            )
    finally:
        logging.disable(previous_logging_disable_level)
    del lm
    clear_device_cache()
    scores = extract_mmlu_scores(results)
    EVAL_RUNS[label] = {"results": results, "scores": scores}
    display_eval_scores(label, scores)
    return results


baseline_results = run_mmlu_eval("baseline")

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


Running baseline MMLU: tasks=['mmlu'], fewshot=5, limit=5, batch=1


## baseline MMLU summary
| metric | value |
| --- | --- |
| acc | 0.5158 |
| acc stderr | 0.0296 |
| acc norm | n/a |
| acc norm stderr | n/a |
| reported tasks | 1 |

In [4]:
import contextlib
import io
import time

import tensor_disentanglement_code.disentanglement_base as DB


# Full MLP pass. Set to a smaller list such as [0, 1, 2, 3] for a quicker trial.
DISENTANGLE_LAYER_INDICES = None
DISENTANGLE_MLP_PROJECTIONS = ("gate_proj", "up_proj", "down_proj")
DISENTANGLE_RANK_FRACTION = 0.25
DISENTANGLE_STEPS = 3
DISENTANGLE_LEARNING_RATE = 0.05
DISENTANGLE_OPTIMIZER = "alt"
DISENTANGLE_SEED = 0


def shape_tuple_from_result(result):
    shape = result.tensor_shape_4d
    return (shape.left_outer, shape.left_inner, shape.right_outer, shape.right_inner)


def shape_tuple_from_weight(weight):
    shape = DB.choose_tensor_shape(weight)
    return (shape.left_outer, shape.left_inner, shape.right_outer, shape.right_inner)


def direct_truncated_svd_weight(weight, keep_rank):
    u, s, vh = torch.linalg.svd(weight, full_matrices=False)
    truncated = (u[:, :keep_rank] * s[:keep_rank]) @ vh[:keep_rank]
    rel_error = torch.linalg.norm(truncated - weight) / torch.linalg.norm(weight)
    return truncated, float(rel_error.item())


def disentangled_dense_weight(linear, *, rank_fraction, steps, learning_rate, optimizer_type, seed):
    original_weight = linear.weight.detach().cpu().to(dtype=torch.float64)
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stderr(stderr_buffer):
        result = DB.disentangle_matrix_via_reshape(
            original_weight,
            steps=steps,
            learning_rate=learning_rate,
            seed=seed,
            device="cpu",
            optimizer_type=optimizer_type,
            rank_selection="fixed_heuristic",
            rank_fraction=rank_fraction,
            log_every=0,
        )
    keep_rank = result.inner.keep_rank
    shape_tuple = shape_tuple_from_result(result)
    q = result.inner.best_q.to(dtype=torch.float64)
    permuted = DB._apply_A(q @ original_weight, shape_tuple)
    u, s, vh = torch.linalg.svd(permuted, full_matrices=False)
    truncated = (u[:, :keep_rank] * s[:keep_rank]) @ vh[:keep_rank]
    reconstructed = q.T @ DB._apply_A_inv(truncated, shape_tuple)
    rel_error = torch.linalg.norm(reconstructed - original_weight) / torch.linalg.norm(original_weight)
    return reconstructed, result, float(rel_error.item()), "tensor_disentanglement"


def selected_mlp_targets(module):
    all_layers = list(module.model.layers)
    layer_indices = DISENTANGLE_LAYER_INDICES
    if layer_indices is None:
        layer_indices = list(range(len(all_layers)))

    targets = []
    for layer_idx in layer_indices:
        mlp = all_layers[layer_idx].mlp
        for projection_name in DISENTANGLE_MLP_PROJECTIONS:
            projection = getattr(mlp, projection_name, None)
            if projection is not None:
                targets.append((layer_idx, projection_name, projection))
    return targets


def apply_mlp_tensor_disentanglement(module):
    targets = selected_mlp_targets(module)
    rows = []
    start = time.time()
    module.eval()
    clear_device_cache()
    print(
        f"Disentangling {len(targets)} MLP linear layers "
        f"with rank_fraction={DISENTANGLE_RANK_FRACTION}, steps={DISENTANGLE_STEPS}, "
        f"optimizer={DISENTANGLE_OPTIMIZER}."
    )

    with torch.inference_mode():
        for target_idx, (layer_idx, projection_name, projection) in enumerate(targets):
            target_name = f"model.layers.{layer_idx}.mlp.{projection_name}"
            print(f"[{target_idx + 1:02d}/{len(targets):02d}] {target_name}")
            target_start = time.time()
            try:
                new_weight, result, rel_error, method = disentangled_dense_weight(
                    projection,
                    rank_fraction=DISENTANGLE_RANK_FRACTION,
                    steps=DISENTANGLE_STEPS,
                    learning_rate=DISENTANGLE_LEARNING_RATE,
                    optimizer_type=DISENTANGLE_OPTIMIZER,
                    seed=DISENTANGLE_SEED + target_idx,
                )
                keep_rank = result.inner.keep_rank
                initial_effective_rank = result.inner.initial_stats.effective_rank
                best_effective_rank = result.inner.best_stats.effective_rank
                error = ""
            except torch.linalg.LinAlgError as exc:
                original_weight = projection.weight.detach().cpu().to(dtype=torch.float64)
                shape_tuple = shape_tuple_from_weight(original_weight)
                keep_rank = DB._default_keep_rank_from_shape(shape_tuple, DISENTANGLE_RANK_FRACTION)
                new_weight, rel_error = direct_truncated_svd_weight(original_weight, keep_rank)
                initial_effective_rank = None
                best_effective_rank = None
                method = "direct_svd_fallback"
                error = f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
            copied_weight = new_weight.to(dtype=projection.weight.dtype).to(device=projection.weight.device)
            projection.weight.copy_(copied_weight)
            rows.append({
                "target": target_name,
                "layer": layer_idx,
                "projection": projection_name,
                "shape": tuple(projection.weight.shape),
                "keep_rank": keep_rank,
                "initial_effective_rank": initial_effective_rank,
                "best_effective_rank": best_effective_rank,
                "relative_error": rel_error,
                "method": method,
                "error": error,
                "seconds": time.time() - target_start,
            })
            clear_device_cache()

    projection_rows = []
    for projection_name in DISENTANGLE_MLP_PROJECTIONS:
        projection_items = [row for row in rows if row["projection"] == projection_name]
        if not projection_items:
            continue
        projection_rows.append([
            projection_name,
            len(projection_items),
            f"{sum(row['relative_error'] for row in projection_items) / len(projection_items):.4f}",
            f"{sum(row['best_effective_rank'] for row in projection_items if row['best_effective_rank'] is not None) / max(1, sum(row['best_effective_rank'] is not None for row in projection_items)):.2f}",
            ", ".join(f"{method}: {sum(row['method'] == method for row in projection_items)}" for method in sorted({row['method'] for row in projection_items})),
            f"{sum(row['seconds'] for row in projection_items):.1f}s",
        ])

    detail_rows = [
        [
            row["target"],
            f"{row['shape']}",
            row["keep_rank"],
            "n/a" if row['initial_effective_rank'] is None else f"{row['initial_effective_rank']:.2f}",
            "n/a" if row['best_effective_rank'] is None else f"{row['best_effective_rank']:.2f}",
            f"{row['relative_error']:.4f}",
            row["method"],
            row["error"],
            f"{row['seconds']:.1f}s",
        ]
        for row in rows
    ]

    display(Markdown(
        "## MLP tensor disentanglement summary\n"
        + f"Processed {len(rows)} MLP linear layers in {time.time() - start:.1f}s. "
        + f"Rank fraction: `{DISENTANGLE_RANK_FRACTION}`, steps: `{DISENTANGLE_STEPS}`, optimizer: `{DISENTANGLE_OPTIMIZER}`.\n\n"
        + md_table(
            ["projection", "count", "mean rel error", "mean best eff rank", "methods", "total time"],
            projection_rows,
        )
        + "\n\n<details><summary>Per-layer details</summary>\n\n"
        + md_table(
            ["target", "shape", "keep rank", "initial eff rank", "best eff rank", "rel error", "method", "error", "time"],
            detail_rows,
        )
        + "\n\n</details>"
    ))
    return rows


disentanglement_report = apply_mlp_tensor_disentanglement(model)

Disentangling 72 MLP linear layers with rank_fraction=0.25, steps=3, optimizer=alt.
[01/72] model.layers.0.mlp.gate_proj
[02/72] model.layers.0.mlp.up_proj
[03/72] model.layers.0.mlp.down_proj
[04/72] model.layers.1.mlp.gate_proj
[05/72] model.layers.1.mlp.up_proj
[06/72] model.layers.1.mlp.down_proj
[07/72] model.layers.2.mlp.gate_proj
[08/72] model.layers.2.mlp.up_proj
[09/72] model.layers.2.mlp.down_proj
[10/72] model.layers.3.mlp.gate_proj
[11/72] model.layers.3.mlp.up_proj
[12/72] model.layers.3.mlp.down_proj
[13/72] model.layers.4.mlp.gate_proj
[14/72] model.layers.4.mlp.up_proj
[15/72] model.layers.4.mlp.down_proj
[16/72] model.layers.5.mlp.gate_proj
[17/72] model.layers.5.mlp.up_proj
[18/72] model.layers.5.mlp.down_proj
[19/72] model.layers.6.mlp.gate_proj
[20/72] model.layers.6.mlp.up_proj
[21/72] model.layers.6.mlp.down_proj
[22/72] model.layers.7.mlp.gate_proj
[23/72] model.layers.7.mlp.up_proj
[24/72] model.layers.7.mlp.down_proj
[25/72] model.layers.8.mlp.gate_proj
[26/72]

## MLP tensor disentanglement summary
Processed 72 MLP linear layers in 1532.6s. Rank fraction: `0.25`, steps: `3`, optimizer: `alt`.

| projection | count | mean rel error | mean best eff rank | methods | total time |
| --- | --- | --- | --- | --- | --- |
| gate_proj | 24 | 0.1329 | 466.76 | direct_svd_fallback: 1, tensor_disentanglement: 23 | 618.4s |
| up_proj | 24 | 0.1070 | 464.88 | tensor_disentanglement: 24 | 642.0s |
| down_proj | 24 | 0.4816 | 878.68 | tensor_disentanglement: 24 | 264.7s |

<details><summary>Per-layer details</summary>

| target | shape | keep rank | initial eff rank | best eff rank | rel error | method | error | time |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| model.layers.0.mlp.gate_proj | (3584, 1024) | 448 | 1135.29 | 476.40 | 0.1307 | tensor_disentanglement |  | 26.3s |
| model.layers.0.mlp.up_proj | (3584, 1024) | 448 | 1153.03 | 459.66 | 0.0958 | tensor_disentanglement |  | 26.8s |
| model.layers.0.mlp.down_proj | (1024, 3584) | 448 | 1148.98 | 883.36 | 0.4835 | tensor_disentanglement |  | 10.9s |
| model.layers.1.mlp.gate_proj | (3584, 1024) | 448 | 1143.04 | 459.92 | 0.0994 | tensor_disentanglement |  | 26.2s |
| model.layers.1.mlp.up_proj | (3584, 1024) | 448 | 1153.22 | 460.89 | 0.0983 | tensor_disentanglement |  | 26.6s |
| model.layers.1.mlp.down_proj | (1024, 3584) | 448 | 1147.64 | 883.16 | 0.4836 | tensor_disentanglement |  | 10.9s |
| model.layers.2.mlp.gate_proj | (3584, 1024) | 448 | 1116.53 | 456.15 | 0.1015 | tensor_disentanglement |  | 27.1s |
| model.layers.2.mlp.up_proj | (3584, 1024) | 448 | 1151.15 | 460.17 | 0.0967 | tensor_disentanglement |  | 26.9s |
| model.layers.2.mlp.down_proj | (1024, 3584) | 448 | 1145.75 | 882.73 | 0.4835 | tensor_disentanglement |  | 11.1s |
| model.layers.3.mlp.gate_proj | (3584, 1024) | 448 | 1104.95 | 457.14 | 0.1072 | tensor_disentanglement |  | 26.5s |
| model.layers.3.mlp.up_proj | (3584, 1024) | 448 | 1148.42 | 461.44 | 0.0993 | tensor_disentanglement |  | 26.9s |
| model.layers.3.mlp.down_proj | (1024, 3584) | 448 | 1146.37 | 884.55 | 0.4844 | tensor_disentanglement |  | 11.1s |
| model.layers.4.mlp.gate_proj | (3584, 1024) | 448 | 1123.43 | 458.71 | 0.1058 | tensor_disentanglement |  | 26.7s |
| model.layers.4.mlp.up_proj | (3584, 1024) | 448 | 1146.34 | 461.47 | 0.0999 | tensor_disentanglement |  | 27.1s |
| model.layers.4.mlp.down_proj | (1024, 3584) | 448 | 1146.17 | 883.63 | 0.4838 | tensor_disentanglement |  | 11.0s |
| model.layers.5.mlp.gate_proj | (3584, 1024) | 448 | 1127.08 | 462.14 | 0.1098 | tensor_disentanglement |  | 26.4s |
| model.layers.5.mlp.up_proj | (3584, 1024) | 448 | 1145.31 | 463.17 | 0.1033 | tensor_disentanglement |  | 27.0s |
| model.layers.5.mlp.down_proj | (1024, 3584) | 448 | 1144.72 | 882.77 | 0.4835 | tensor_disentanglement |  | 11.1s |
| model.layers.6.mlp.gate_proj | (3584, 1024) | 448 | 1132.45 | 465.66 | 0.1162 | tensor_disentanglement |  | 26.4s |
| model.layers.6.mlp.up_proj | (3584, 1024) | 448 | 1142.79 | 465.50 | 0.1083 | tensor_disentanglement |  | 26.7s |
| model.layers.6.mlp.down_proj | (1024, 3584) | 448 | 1140.39 | 879.91 | 0.4825 | tensor_disentanglement |  | 11.0s |
| model.layers.7.mlp.gate_proj | (3584, 1024) | 448 | 1131.22 | 465.13 | 0.1155 | tensor_disentanglement |  | 26.6s |
| model.layers.7.mlp.up_proj | (3584, 1024) | 448 | 1138.74 | 464.48 | 0.1068 | tensor_disentanglement |  | 26.8s |
| model.layers.7.mlp.down_proj | (1024, 3584) | 448 | 1137.09 | 878.92 | 0.4821 | tensor_disentanglement |  | 11.0s |
| model.layers.8.mlp.gate_proj | (3584, 1024) | 448 | 1133.71 | 466.82 | 0.1167 | tensor_disentanglement |  | 26.4s |
| model.layers.8.mlp.up_proj | (3584, 1024) | 448 | 1138.63 | 466.50 | 0.1107 | tensor_disentanglement |  | 26.9s |
| model.layers.8.mlp.down_proj | (1024, 3584) | 448 | 1139.87 | 879.73 | 0.4825 | tensor_disentanglement |  | 11.0s |
| model.layers.9.mlp.gate_proj | (3584, 1024) | 448 | 1136.30 | 468.55 | 0.1179 | tensor_disentanglement |  | 26.4s |
| model.layers.9.mlp.up_proj | (3584, 1024) | 448 | 1139.69 | 468.38 | 0.1141 | tensor_disentanglement |  | 26.8s |
| model.layers.9.mlp.down_proj | (1024, 3584) | 448 | 1135.60 | 876.13 | 0.4808 | tensor_disentanglement |  | 11.0s |
| model.layers.10.mlp.gate_proj | (3584, 1024) | 448 | 1133.03 | 469.56 | 0.1220 | tensor_disentanglement |  | 26.6s |
| model.layers.10.mlp.up_proj | (3584, 1024) | 448 | 1141.98 | 467.41 | 0.1124 | tensor_disentanglement |  | 26.8s |
| model.layers.10.mlp.down_proj | (1024, 3584) | 448 | 1139.36 | 877.01 | 0.4812 | tensor_disentanglement |  | 11.0s |
| model.layers.11.mlp.gate_proj | (3584, 1024) | 448 | 1125.25 | 466.33 | 0.1180 | tensor_disentanglement |  | 26.4s |
| model.layers.11.mlp.up_proj | (3584, 1024) | 448 | 1141.60 | 467.29 | 0.1122 | tensor_disentanglement |  | 26.7s |
| model.layers.11.mlp.down_proj | (1024, 3584) | 448 | 1133.69 | 874.12 | 0.4800 | tensor_disentanglement |  | 11.0s |
| model.layers.12.mlp.gate_proj | (3584, 1024) | 448 | n/a | n/a | 0.4533 | direct_svd_fallback | _LinAlgError: linalg.svd: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated singular values (error code: 12). | 10.1s |
| model.layers.12.mlp.up_proj | (3584, 1024) | 448 | 1143.40 | 467.66 | 0.1128 | tensor_disentanglement |  | 26.9s |
| model.layers.12.mlp.down_proj | (1024, 3584) | 448 | 1136.83 | 875.69 | 0.4805 | tensor_disentanglement |  | 11.0s |
| model.layers.13.mlp.gate_proj | (3584, 1024) | 448 | 1124.05 | 464.28 | 0.1136 | tensor_disentanglement |  | 26.6s |
| model.layers.13.mlp.up_proj | (3584, 1024) | 448 | 1144.31 | 466.81 | 0.1110 | tensor_disentanglement |  | 26.7s |
| model.layers.13.mlp.down_proj | (1024, 3584) | 448 | 1136.45 | 876.81 | 0.4811 | tensor_disentanglement |  | 11.0s |
| model.layers.14.mlp.gate_proj | (3584, 1024) | 448 | 1111.73 | 464.39 | 0.1157 | tensor_disentanglement |  | 26.4s |
| model.layers.14.mlp.up_proj | (3584, 1024) | 448 | 1144.94 | 465.78 | 0.1090 | tensor_disentanglement |  | 26.5s |
| model.layers.14.mlp.down_proj | (1024, 3584) | 448 | 1145.39 | 880.64 | 0.4824 | tensor_disentanglement |  | 11.0s |
| model.layers.15.mlp.gate_proj | (3584, 1024) | 448 | 1115.78 | 462.47 | 0.1163 | tensor_disentanglement |  | 26.4s |
| model.layers.15.mlp.up_proj | (3584, 1024) | 448 | 1141.79 | 463.86 | 0.1060 | tensor_disentanglement |  | 26.8s |
| model.layers.15.mlp.down_proj | (1024, 3584) | 448 | 1130.72 | 873.52 | 0.4789 | tensor_disentanglement |  | 11.2s |
| model.layers.16.mlp.gate_proj | (3584, 1024) | 448 | 1123.20 | 529.93 | 0.2081 | tensor_disentanglement |  | 26.4s |
| model.layers.16.mlp.up_proj | (3584, 1024) | 448 | 1144.43 | 466.09 | 0.1096 | tensor_disentanglement |  | 26.8s |
| model.layers.16.mlp.down_proj | (1024, 3584) | 448 | 1133.85 | 875.22 | 0.4799 | tensor_disentanglement |  | 11.0s |
| model.layers.17.mlp.gate_proj | (3584, 1024) | 448 | 1120.96 | 481.74 | 0.1449 | tensor_disentanglement |  | 26.3s |
| model.layers.17.mlp.up_proj | (3584, 1024) | 448 | 1145.29 | 465.58 | 0.1087 | tensor_disentanglement |  | 26.8s |
| model.layers.17.mlp.down_proj | (1024, 3584) | 448 | 1135.24 | 876.16 | 0.4805 | tensor_disentanglement |  | 11.0s |
| model.layers.18.mlp.gate_proj | (3584, 1024) | 448 | 1114.32 | 458.18 | 0.1101 | tensor_disentanglement |  | 26.4s |
| model.layers.18.mlp.up_proj | (3584, 1024) | 448 | 1146.24 | 465.16 | 0.1077 | tensor_disentanglement |  | 26.8s |
| model.layers.18.mlp.down_proj | (1024, 3584) | 448 | 1137.27 | 876.88 | 0.4806 | tensor_disentanglement |  | 11.0s |
| model.layers.19.mlp.gate_proj | (3584, 1024) | 448 | 1115.43 | 480.53 | 0.1454 | tensor_disentanglement |  | 26.3s |
| model.layers.19.mlp.up_proj | (3584, 1024) | 448 | 1148.84 | 466.85 | 0.1111 | tensor_disentanglement |  | 26.7s |
| model.layers.19.mlp.down_proj | (1024, 3584) | 448 | 1138.48 | 877.67 | 0.4808 | tensor_disentanglement |  | 11.1s |
| model.layers.20.mlp.gate_proj | (3584, 1024) | 448 | 1121.62 | 460.48 | 0.1097 | tensor_disentanglement |  | 26.4s |
| model.layers.20.mlp.up_proj | (3584, 1024) | 448 | 1151.12 | 466.60 | 0.1102 | tensor_disentanglement |  | 26.6s |
| model.layers.20.mlp.down_proj | (1024, 3584) | 448 | 1140.92 | 879.58 | 0.4819 | tensor_disentanglement |  | 11.0s |
| model.layers.21.mlp.gate_proj | (3584, 1024) | 448 | 1118.40 | 457.65 | 0.1067 | tensor_disentanglement |  | 26.5s |
| model.layers.21.mlp.up_proj | (3584, 1024) | 448 | 1151.47 | 465.20 | 0.1074 | tensor_disentanglement |  | 26.7s |
| model.layers.21.mlp.down_proj | (1024, 3584) | 448 | 1141.15 | 879.42 | 0.4817 | tensor_disentanglement |  | 11.0s |
| model.layers.22.mlp.gate_proj | (3584, 1024) | 448 | 1116.94 | 456.60 | 0.1061 | tensor_disentanglement |  | 26.4s |
| model.layers.22.mlp.up_proj | (3584, 1024) | 448 | 1151.58 | 465.46 | 0.1078 | tensor_disentanglement |  | 26.5s |
| model.layers.22.mlp.down_proj | (1024, 3584) | 448 | 1140.63 | 878.78 | 0.4813 | tensor_disentanglement |  | 11.0s |
| model.layers.23.mlp.gate_proj | (3584, 1024) | 448 | 1091.22 | 446.65 | 0.0993 | tensor_disentanglement |  | 26.4s |
| model.layers.23.mlp.up_proj | (3584, 1024) | 448 | 1148.84 | 465.76 | 0.1092 | tensor_disentanglement |  | 26.4s |
| model.layers.23.mlp.down_proj | (1024, 3584) | 448 | 1131.45 | 871.84 | 0.4779 | tensor_disentanglement |  | 11.0s |

</details>

In [5]:
mlp_disentangled_results = run_mmlu_eval("mlp_disentangled")


def compare_eval_runs(left_label="baseline", right_label="mlp_disentangled"):
    left = EVAL_RUNS[left_label]["scores"]
    right = EVAL_RUNS[right_label]["scores"]
    rows = []
    for metric in ["acc", "acc_norm"]:
        left_value = left.get(metric)
        right_value = right.get(metric)
        if left_value is None or right_value is None:
            delta = None
        else:
            delta = right_value - left_value
        rows.append([metric, fmt_float(left_value), fmt_float(right_value), fmt_float(delta)])
    display(Markdown(
        "## Baseline vs MLP-disentangled MMLU\n"
        + md_table(["metric", left_label, right_label, "delta"], rows)
    ))


compare_eval_runs()

Running mlp_disentangled MMLU: tasks=['mmlu'], fewshot=5, limit=5, batch=1


## mlp_disentangled MMLU summary
| metric | value |
| --- | --- |
| acc | 0.4246 |
| acc stderr | 0.0281 |
| acc norm | n/a |
| acc norm stderr | n/a |
| reported tasks | 1 |

## Baseline vs MLP-disentangled MMLU
| metric | baseline | mlp_disentangled | delta |
| --- | --- | --- | --- |
| acc | 0.5158 | 0.4246 | -0.0912 |
| acc_norm | n/a | n/a | n/a |